In [1]:
import os
import difflib


In [2]:

# Define the base paths
base_dir = "/home/id.aau.dk/kr75cs/MultiClinNer/Multi-Head-CRF/dataset/MultiClinAI-training_data_v1.1-260225/MultiClinNER/MultiClinNER-es/MultiClinNER-es-train"


In [3]:

directories = {
    "symptom": os.path.join(base_dir, "MultiClinNER-es-train-symptom/txt"),
    "disease": os.path.join(base_dir, "MultiClinNER-es-train-disease/txt"),
    "procedure": os.path.join(base_dir, "MultiClinNER-es-train-procedure/txt")
}

# Dictionary to hold file paths grouped by their base ID
# Format: {'0001': {'symptom': 'path/...', 'disease': 'path/...', 'procedure': 'path/...'}}
files_by_id = {}

# 1. Gather all files and group them by ID
for label, folder_path in directories.items():
    if not os.path.exists(folder_path):
        print(f"Warning: Directory not found -> {folder_path}")
        continue
        
    for filename in os.listdir(folder_path):
        if filename.endswith(".txt"):
            # Extract the ID from the filename (e.g., "MultiClinNER-es-train-symptom-0001.txt" -> "0001")
            file_id = filename.split('-')[-1].replace('.txt', '')
            
            if file_id not in files_by_id:
                files_by_id[file_id] = {}
            files_by_id[file_id][label] = os.path.join(folder_path, filename)

# 2. Compare the files for each ID
print(f"Found {len(files_by_id)} unique file IDs. Starting comparison...\n")
print("-" * 50)

for file_id, paths in sorted(files_by_id.items()):
    labels_present = list(paths.keys())
    
    # We need at least 2 files to make a comparison
    if len(labels_present) < 2:
        print(f"ID {file_id}: Only found in {labels_present[0]}, nothing to compare.")
        continue

    # Read the contents of all available files for this ID
    contents = {}
    for label in labels_present:
        with open(paths[label], 'r', encoding='utf-8') as f:
            contents[label] = f.readlines()

    # Use the first available file as the base for comparison
    base_label = labels_present[0]
    base_content = contents[base_label]
    
    mismatches_found = False
    
    # Compare the base file against the others
    for other_label in labels_present[1:]:
        if base_content != contents[other_label]:
            mismatches_found = True
            print(f"⚠️ DIFFERENCE FOUND in ID {file_id}: '{base_label}' vs '{other_label}'")
            
            # Print a detailed diff showing exactly what changed
            diff = difflib.unified_diff(
                base_content, 
                contents[other_label], 
                fromfile=f"{base_label}-{file_id}", 
                tofile=f"{other_label}-{file_id}", 
                lineterm=''
            )
            for line in diff:
                print(line.strip('\n'))
            print("\n")

    if not mismatches_found:
        # Optional: Uncomment the next line if you want it to explicitly tell you when files are identical
        # print(f"✅ ID {file_id}: All corresponding text files are perfectly identical.")
        pass

print("-" * 50)
print("Comparison complete.")

Found 1258 unique file IDs. Starting comparison...

--------------------------------------------------
⚠️ DIFFERENCE FOUND in ID 0591: 'symptom' vs 'procedure'
--- symptom-0591
+++ procedure-0591
@@ -1,4 +1,4 @@
 Antecedentes: Hombre de 45 años con antecedentes de alcoholismo crónico que se acompaña de pancreatitis crónica y diabetes mellitus insulino dependiente, que presenta una ingesta alimentaria muy escasa. Múltiples ingresos hospitalarios por reagudizaciones de su pancreatitis crónica. En 2002 es intervenido para una extirpación de un pseudoquiste pancreático que se complica y finaliza con una quistogastrostomía que se resuelve favorablemente. En febrero de 2003 Hay nuevo ingreso por colecistitis enfisematosa con absceso perivesicular complicado por lo que se coloca colecistostomía con tubo de Pezzer para realizar colecistectomía en un segundo tiempo quirúrgico. Se da de alta con un drenaje por tubo de colecistostomia de aproximadamente 500 ml. En noviembre de 2003 acude a consul

In [4]:
import os
import shutil

# Define the base paths
base_dir = "/home/id.aau.dk/kr75cs/MultiClinNer/Multi-Head-CRF/dataset/MultiClinAI-training_data_v1.1-260225/MultiClinNER/MultiClinNER-es/MultiClinNER-es-train"

directories = [
    os.path.join(base_dir, "MultiClinNER-es-train-symptom/txt"),
    os.path.join(base_dir, "MultiClinNER-es-train-disease/txt"),
    os.path.join(base_dir, "MultiClinNER-es-train-procedure/txt")
]

# Define and create the new destination directory
dest_dir = os.path.join('/home/id.aau.dk/kr75cs/MultiClinNer/Multi-Head-CRF/dataset', "documents-es")
os.makedirs(dest_dir, exist_ok=True)

# Keep track of which IDs we have already copied
copied_ids = set()
files_copied = 0

print(f"Creating directory and copying files to: \n{dest_dir}\n")

# Loop through the folders and copy files
for folder_path in directories:
    if not os.path.exists(folder_path):
        print(f"Warning: Directory not found -> {folder_path}")
        continue
        
    for filename in os.listdir(folder_path):
        if filename.endswith(".txt"):
            # Extract the ID (e.g., "MultiClinNER-es-train-symptom-0001.txt" -> "0001")
            file_id = filename.split('-')[-1].replace('.txt', '')
            
            # If we haven't copied a file for this ID yet, copy it now
            if file_id not in copied_ids:
                source_path = os.path.join(folder_path, filename)
                
                # Create the new filename
                dest_filename = f"es-train-{file_id}.txt"
                dest_path = os.path.join(dest_dir, dest_filename)
                
                # Copy the file (copy2 preserves original metadata like creation date)
                shutil.copy2(source_path, dest_path)
                
                # Mark this ID as handled
                copied_ids.add(file_id)
                files_copied += 1

print("-" * 50)
print(f"✅ Success! Copied {files_copied} unique files into the 'documents-es' directory.")

Creating directory and copying files to: 
/home/id.aau.dk/kr75cs/MultiClinNer/Multi-Head-CRF/dataset/documents-es

--------------------------------------------------
✅ Success! Copied 1258 unique files into the 'documents-es' directory.


In [5]:
import os
import csv

# Directory containing the TSV files
base_dir = "/home/id.aau.dk/kr75cs/MultiClinNer/Multi-Head-CRF/dataset/clean_ner"

# List of your TSV files
tsv_files = [
    "MultiClinNER-es-train-disease.tsv",
    "MultiClinNER-es-train-procedure.tsv",
    "MultiClinNER-es-train-symptom.tsv"
]

print("Starting TSV update...\n")
print("-" * 50)

for file_name in tsv_files:
    input_path = os.path.join(base_dir, file_name)
    
    # Create an output path for the updated file
    output_name = file_name.replace(".tsv", "_updated.tsv")
    output_path = os.path.join(base_dir, output_name)
    
    if not os.path.exists(input_path):
        print(f"⚠️ Warning: File not found -> {input_path}")
        continue

    # Keep track of how many rows we process
    row_count = 0

    with open(input_path, 'r', encoding='utf-8') as infile, \
         open(output_path, 'w', encoding='utf-8', newline='') as outfile:
        
        # We use a tab delimiter for TSV files
        reader = csv.reader(infile, delimiter='\t')
        writer = csv.writer(outfile, delimiter='\t')
        
        # Read and write the header row exactly as it is
        try:
            header = next(reader)
            writer.writerow(header)
        except StopIteration:
            print(f"File {file_name} is empty. Skipping.")
            continue
        
        # Process the rest of the rows
        for row in reader:
            if not row:  # Skip any completely blank lines
                continue
                
            old_filename = row[0]
            
            # Extract the ID (e.g., "MultiClinNER-es-train-procedure-1254" -> "1254")
            file_id = old_filename.split('-')[-1]
            
            # Create the new filename to match your documents folder
            new_filename = f"es-train-{file_id}"
            
            # Update the row and write it to the new file
            row[0] = new_filename
            writer.writerow(row)
            row_count += 1
            
    print(f"✅ Processed {row_count} rows in {file_name}")
    print(f"   Saved to: {output_name}")

print("-" * 50)
print("Update complete!")

Starting TSV update...

--------------------------------------------------
✅ Processed 26296 rows in MultiClinNER-es-train-disease.tsv
   Saved to: MultiClinNER-es-train-disease_updated.tsv
✅ Processed 28137 rows in MultiClinNER-es-train-procedure.tsv
   Saved to: MultiClinNER-es-train-procedure_updated.tsv
✅ Processed 29074 rows in MultiClinNER-es-train-symptom.tsv
   Saved to: MultiClinNER-es-train-symptom_updated.tsv
--------------------------------------------------
Update complete!


In [6]:
import os
import csv

# Define your paths
txt_dir = "/home/id.aau.dk/kr75cs/MultiClinNer/Multi-Head-CRF/dataset/documents-es"

# Replace this with the path to the TSV file you want to verify
# If you combined them, point to the combined file. Otherwise, you can loop through the updated ones.
tsv_path = "/home/id.aau.dk/kr75cs/MultiClinNer/Multi-Head-CRF/dataset/merged_data_ner_es_comp_data_train.tsv" 

print(f"Starting verification for: {os.path.basename(tsv_path)}\n")
print("-" * 50)

# Counters for our final report
total_annotations = 0
exact_matches = 0
mismatches = []
missing_files = set()

# Cache the text files in memory so we don't read from the disk thousands of times for the same file
text_cache = {}

with open(tsv_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f, delimiter='\t')
    
    for row_num, row in enumerate(reader, start=2): # start=2 because row 1 is the header
        total_annotations += 1
        
        filename = row['filename']
        label = row['label']
        start_span = int(row['start_span'])
        end_span = int(row['end_span'])
        expected_text = row['text']
        
        txt_filepath = os.path.join(txt_dir, f"{filename}.txt")
        
        # Load the text file if we haven't already
        if filename not in text_cache:
            if not os.path.exists(txt_filepath):
                missing_files.add(filename)
                continue
            with open(txt_filepath, 'r', encoding='utf-8') as txt_file:
                text_cache[filename] = txt_file.read()
                
        # Extract the text using the spans
        document_text = text_cache[filename]
        extracted_text = document_text[start_span:end_span]
        
        # Compare
        if extracted_text == expected_text:
            exact_matches += 1
        else:
            mismatches.append({
                'row': row_num,
                'filename': filename,
                'label': label,
                'expected': expected_text,
                'extracted': extracted_text,
                'spans': f"[{start_span}:{end_span}]"
            })

# Print the Results
print(f"Total annotations checked: {total_annotations}")
print(f"✅ Exact matches: {exact_matches}")
print(f"❌ Mismatches: {len(mismatches)}")

if missing_files:
    print(f"⚠️ Missing text files for {len(missing_files)} documents.")

if mismatches:
    print("\n" + "=" * 50)
    print("DETAILED MISMATCH REPORT (Showing first 10)")
    print("=" * 50)
    for m in mismatches[:10]:
        print(f"Row {m['row']} in {m['filename']} ({m['label']}):")
        print(f"  Spans:     {m['spans']}")
        print(f"  Expected:  '{m['expected']}'")
        print(f"  Extracted: '{m['extracted']}'\n")

Starting verification for: merged_data_ner_es_comp_data_train.tsv

--------------------------------------------------
Total annotations checked: 83507
✅ Exact matches: 83507
❌ Mismatches: 0


In [7]:
import os
import csv

# Define paths
input_tsv = "/home/id.aau.dk/kr75cs/MultiClinNer/Multi-Head-CRF/dataset/merged_data_ner_es_comp_data_full.tsv"
base_dir = os.path.dirname(input_tsv)

train_tsv = os.path.join(base_dir, "merged_data_ner_es_comp_data_train.tsv")
val_tsv = os.path.join(base_dir, "merged_data_ner_es_comp_data_val.tsv")

print("Reading the dataset and grouping by document...")
print("-" * 50)

# Step 1: Read all rows and keep track of unique document filenames in order
rows = []
unique_docs = []

with open(input_tsv, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f, delimiter='\t')
    header = reader.fieldnames
    
    for row in reader:
        rows.append(row)
        filename = row['filename']
        
        # Keep track of the order documents appear
        if not unique_docs or unique_docs[-1] != filename:
            if filename not in unique_docs:
                unique_docs.append(filename)

total_docs = len(unique_docs)

# Ensure we have enough documents to split as requested
if total_docs < 258:
    print(f"Error: Only found {total_docs} documents, which is less than the requested 258 validation docs.")
    exit()

# Step 2: Split the documents (last 258 for val, the rest for train)
train_docs = set(unique_docs[:-258])
val_docs = set(unique_docs[-258:])

print(f"Total unique documents found: {total_docs}")
print(f"Allocating {len(train_docs)} documents for training.")
print(f"Allocating {len(val_docs)} documents for validation.")

# Step 3: Write the split data into two new TSV files
train_ann_count = 0
val_ann_count = 0

with open(train_tsv, 'w', encoding='utf-8', newline='') as f_train, \
     open(val_tsv, 'w', encoding='utf-8', newline='') as f_val:
    
    writer_train = csv.DictWriter(f_train, fieldnames=header, delimiter='\t')
    writer_val = csv.DictWriter(f_val, fieldnames=header, delimiter='\t')
    
    writer_train.writeheader()
    writer_val.writeheader()
    
    for row in rows:
        if row['filename'] in train_docs:
            writer_train.writerow(row)
            train_ann_count += 1
        elif row['filename'] in val_docs:
            writer_val.writerow(row)
            val_ann_count += 1

print("-" * 50)
print(f"✅ Success!")
print(f"Training set:   {train_ann_count} annotations saved to train_es.tsv")
print(f"Validation set: {val_ann_count} annotations saved to val_es.tsv")

Reading the dataset and grouping by document...
--------------------------------------------------
Total unique documents found: 1258
Allocating 1000 documents for training.
Allocating 258 documents for validation.
--------------------------------------------------
✅ Success!
Training set:   55015 annotations saved to train_es.tsv
Validation set: 28492 annotations saved to val_es.tsv


In [8]:
import pandas as pd
import os

# Point this to the Dutch (or any failing language) TSV and Text folder
tsv_path = "/home/id.aau.dk/kr75cs/MultiClinNer/Multi-Head-CRF/dataset/processed_multilingual/nl/merged_data_nl.tsv"
txt_dir = "/home/id.aau.dk/kr75cs/MultiClinNer/Multi-Head-CRF/dataset/processed_multilingual/documents-nl"

df = pd.read_csv(tsv_path, sep='\t')
mismatches = 0

print("🔍 FIRST 10 MISMATCHES IN NL:")
print("-" * 60)

for _, row in df.iterrows():
    filename = row['filename']
    start, end = int(row['start_span']), int(row['end_span'])
    expected = str(row['text'])
    
    txt_filepath = os.path.join(txt_dir, f"{filename}.txt")
    if not os.path.exists(txt_filepath):
        continue
        
    with open(txt_filepath, 'r') as f:
        text = f.read()
        
    extracted = text[start:end]
    
    if extracted != expected:
        mismatches += 1
        
        # Grab a little context around the error to see the shift
        context_start = max(0, start - 10)
        context_end = min(len(text), end + 10)
        context = text[context_start:context_end].replace('\n', '\\n')
        
        print(f"File:      {filename}")
        print(f"Spans:     [{start}:{end}]")
        print(f"Expected:  '{expected}'")
        print(f"Extracted: '{extracted}'")
        print(f"Context:   '{context}'")
        print("-" * 60)
        print(text)
        print("-" * 60)

        
    if mismatches >= 50:
        break

🔍 FIRST 10 MISMATCHES IN NL:
------------------------------------------------------------
File:      nl-train-0001
Spans:     [339:360]
Expected:  'Laboratoriumonderzoek'
Extracted: ' Laboratoriumonderzoe'
Context:   'rikkeling. Laboratoriumonderzoek: verhoog'
------------------------------------------------------------
We presenteren de casus van een 43-jarige man met diabetes in de anamnese, die werd opgenomen voor matige acute pancreatitis. Twee weken later meldde hij acute pijn in de linker hemiabdomen en misselijkheid, zonder andere symptomen. Bij onderzoek werd een grote bezetting van de linker hemiabdomen gepalpeerd zonder peritoneale prikkeling. Laboratoriumonderzoek: verhoogde triglyceriden en lipase, andere biochemie, hemogram en stolling normaal. Tijdens het onderzoek kreeg hij syncope en zijn hematocriet daalde van 40 naar 21%, waardoor een transfusie van 6 rode bloedcelconcentraten nodig was, en hij bleef stabiel. Beeldvormende tests toonden een hemorragische pancreas pseu

In [9]:
import os

def check_text_differences():
    # Base path for the raw Dutch training data
    base_dir = "/home/id.aau.dk/kr75cs/MultiClinNer/Multi-Head-CRF/dataset/MultiClinAI-training_data_v1.1-260225/MultiClinNER/MultiClinNER-nl/MultiClinNER-nl-train"
    
    entities = ["symptom", "disease", "procedure"]
    
    # We will check the first 10 documents, including our suspected troublemaker "0002"
    file_ids_to_check = [f"{i:04d}" for i in range(1, 11)] 
    
    print("🔍 VALIDATING TEXT FILE CONSISTENCY ACROSS ENTITY FOLDERS (NL)...")
    print("-" * 70)
    
    for file_id in file_ids_to_check:
        contents = {}
        paths_found = []
        
        # Read the text file from all three entity folders
        for entity in entities:
            # Construct the exact filename based on the dataset's naming convention
            filename = f"MultiClinNER-nl-train-{entity}-{file_id}.txt"
            filepath = os.path.join(base_dir, f"MultiClinNER-nl-train-{entity}", "txt", filename)
            
            if os.path.exists(filepath):
                # Using standard utf-8 since we are just comparing raw strings
                with open(filepath, 'r', encoding='utf-8') as f:
                    contents[entity] = f.read()
                    paths_found.append(entity)
                    
        # If we didn't find at least 2 files to compare, skip
        if len(paths_found) < 2:
            continue
            
        # Compare the other entities against the first one we found (usually 'symptom')
        baseline_entity = paths_found[0]
        baseline_text = contents[baseline_entity]
        
        mismatch_found = False
        
        for other_entity in paths_found[1:]:
            other_text = contents[other_entity]
            
            if baseline_text != other_text:
                mismatch_found = True
                print(f"🚨 MISMATCH FOUND IN ID {file_id}: '{baseline_entity}' text != '{other_entity}' text!")
                print(f"   - Length of {baseline_entity} text: {len(baseline_text)} characters")
                print(f"   - Length of {other_entity} text: {len(other_text)} characters")
                print(f"   - Difference in length: {abs(len(baseline_text) - len(other_text))} characters\n")
                
        if not mismatch_found:
            print(f"✅ ID {file_id}: All text files are perfectly identical across folders.")

    print("-" * 70)

if __name__ == "__main__":
    check_text_differences()

🔍 VALIDATING TEXT FILE CONSISTENCY ACROSS ENTITY FOLDERS (NL)...
----------------------------------------------------------------------
🚨 MISMATCH FOUND IN ID 0001: 'symptom' text != 'disease' text!
   - Length of symptom text: 954 characters
   - Length of disease text: 959 characters
   - Difference in length: 5 characters

🚨 MISMATCH FOUND IN ID 0001: 'symptom' text != 'procedure' text!
   - Length of symptom text: 954 characters
   - Length of procedure text: 953 characters
   - Difference in length: 1 characters

🚨 MISMATCH FOUND IN ID 0002: 'symptom' text != 'disease' text!
   - Length of symptom text: 1396 characters
   - Length of disease text: 1424 characters
   - Difference in length: 28 characters

🚨 MISMATCH FOUND IN ID 0002: 'symptom' text != 'procedure' text!
   - Length of symptom text: 1396 characters
   - Length of procedure text: 1424 characters
   - Difference in length: 28 characters

✅ ID 0003: All text files are perfectly identical across folders.
🚨 MISMATCH FOUND

In [1]:
import os
import glob
import pandas as pd
import re
from difflib import SequenceMatcher

# ==========================================
# CONFIGURATION
# ==========================================
DATASET_ROOT = "/home/id.aau.dk/kr75cs/MultiClinNer/Multi-Head-CRF/dataset"
RAW_DATA_DIR = os.path.join(DATASET_ROOT, "MultiClinAI-training_data_v1.1-260225/MultiClinNER")
ENTITIES = ["symptom", "disease", "procedure"]

def realign_span(old_text, new_text, old_start, old_end, entity_string):
    """
    Attempts to find the entity in the new text. 
    Returns: (new_span, is_exact_match)
    """
    try:
        matches = list(re.finditer(re.escape(str(entity_string)), new_text))
    except: return None, False
    
    if not matches: return None, False
    
    # Check for Exact Match (Same indices as original)
    for m in matches:
        if m.start() == old_start and m.end() == old_end:
            return (m.start(), m.end()), True

    # Contextual Realignment if no exact match found
    if len(matches) == 1: return matches[0].span(), False

    context_size = 30
    old_context = old_text[max(0, old_start - context_size) : min(len(old_text), old_end + context_size)]
    best_match, best_score = None, -1
    for m in matches:
        new_start, new_end = m.span()
        new_context = new_text[max(0, new_start - context_size) : min(len(new_text), new_end + context_size)]
        score = SequenceMatcher(None, old_context, new_context).ratio()
        if score > best_score:
            best_score, best_match = score, (new_start, new_end)
            
    return (best_match, False) if best_score > 0.6 else (None, False)

def main():
    print(f"{'='*120}\n🌍 MULTILINGUAL CONSOLIDATION: 3-WAY CONSENSUS vs. CONTEXTUAL RESCUE\n{'='*120}")
    lang_dirs = sorted(glob.glob(os.path.join(RAW_DATA_DIR, "MultiClinNER-*")))
    final_results = []

    for lang_dir in lang_dirs:
        lang = os.path.basename(lang_dir).split('-')[-1]
        print(f"\n▶️ LANGUAGE: {lang.upper()}")
        
        # Load all TSVs
        dfs = {}
        original_total = 0
        for ent in ENTITIES:
            path = os.path.join(lang_dir, f"MultiClinNER-{lang}-train", f"MultiClinNER-{lang}-train-{ent}", "tsv", f"MultiClinNER-{lang}-train-{ent}.tsv")
            if os.path.exists(path):
                df = pd.read_csv(path, sep='\t', names=['filename', 'label', 'start', 'end', 'text'], skiprows=1)
                df['doc_id'] = df['filename'].apply(lambda x: str(x).split('-')[-1])
                dfs[ent] = df
                original_total += len(df)

        if len(dfs) < 3: continue

        for anchor in ENTITIES:
            stats = {"gold_consensus": 0, "rescued": 0, "lost": 0}
            others = [e for e in ENTITIES if e != anchor]
            unique_ids = sorted(dfs[anchor]['doc_id'].unique())

            for doc_id in unique_ids:
                txts = {}
                for ent in ENTITIES:
                    p = os.path.join(lang_dir, f"MultiClinNER-{lang}-train", f"MultiClinNER-{lang}-train-{ent}", "txt", f"MultiClinNER-{lang}-train-{ent}-{doc_id}.txt")
                    if os.path.exists(p):
                        with open(p, 'r', encoding='utf-8-sig', newline='') as f: txts[ent] = f.read()
                
                if len(txts) < 3: continue
                anchor_text = txts[anchor]

                # Check ALL 3 entity dataframes against the Anchor Text
                for current_ent in ENTITIES:
                    ent_df = dfs[current_ent][dfs[current_ent]['doc_id'] == doc_id]
                    source_text = txts[current_ent]

                    for _, row in ent_df.iterrows():
                        new_span, is_exact = realign_span(source_text, anchor_text, int(row['start']), int(row['end']), row['text'])
                        
                        if is_exact:
                            stats["gold_consensus"] += 1
                        elif new_span:
                            stats["rescued"] += 1
                        else:
                            stats["lost"] += 1

            total_valid = stats["gold_consensus"] + stats["rescued"]
            retention = (total_valid / original_total) * 100
            
            print(f"   ⚓ Anchor: {anchor:<10} | Gold (Match All): {stats['gold_consensus']:>5} | Rescued: {stats['rescued']:>5} | Valid: {total_valid:>6} | {retention:>5.2f}%")
            
            final_results.append({
                "Lang": lang.upper(), "Anchor": anchor, "Expected": original_total, 
                "Gold Consensus": stats["gold_consensus"], "Rescued": stats["rescued"], 
                "Lost": stats["lost"], "Retention %": round(retention, 2)
            })

    print(f"\n{'='*120}\n📊 FINAL CONSOLIDATION SUMMARY (GOLD = EXACT MATCH ACROSS ALL 3)\n{'='*120}")
    summary_df = pd.DataFrame(final_results)
    summary_df = summary_df.sort_values(by=["Lang", "Retention %"], ascending=[True, False])
    print(summary_df.to_string(index=False))

if __name__ == "__main__":
    main()

/tmp/ipykernel_681839/1086040608.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


🌍 MULTILINGUAL CONSOLIDATION: 3-WAY CONSENSUS vs. CONTEXTUAL RESCUE

▶️ LANGUAGE: CZ
   ⚓ Anchor: symptom    | Gold (Match All): 52428 | Rescued: 26518 | Valid:  78946 | 97.34%
   ⚓ Anchor: disease    | Gold (Match All): 53805 | Rescued: 24355 | Valid:  78160 | 96.37%
   ⚓ Anchor: procedure  | Gold (Match All): 54489 | Rescued: 23413 | Valid:  77902 | 96.06%

▶️ LANGUAGE: EN
   ⚓ Anchor: symptom    | Gold (Match All): 71775 | Rescued:  6781 | Valid:  78556 | 99.04%
   ⚓ Anchor: disease    | Gold (Match All): 71776 | Rescued:  6706 | Valid:  78482 | 98.95%
   ⚓ Anchor: procedure  | Gold (Match All): 72089 | Rescued:  6443 | Valid:  78532 | 99.01%

▶️ LANGUAGE: ES
   ⚓ Anchor: symptom    | Gold (Match All): 83410 | Rescued:     0 | Valid:  83410 | 99.88%
   ⚓ Anchor: disease    | Gold (Match All): 83507 | Rescued:     0 | Valid:  83507 | 100.00%
   ⚓ Anchor: procedure  | Gold (Match All): 83502 | Rescued:     0 | Valid:  83502 | 99.99%

▶️ LANGUAGE: IT
   ⚓ Anchor: symptom    | Gold (Mat

In [2]:
import os
import glob
import pandas as pd
import re
from difflib import SequenceMatcher

# ==========================================
# CONFIGURATION
# ==========================================
DATASET_ROOT = "/home/id.aau.dk/kr75cs/MultiClinNer/Multi-Head-CRF/dataset"
RAW_DATA_DIR = os.path.join(DATASET_ROOT, "MultiClinAI-training_data_v1.1-260225/MultiClinNER")
ENTITIES = ["symptom", "disease", "procedure"]

def realign_span(old_text, new_text, old_start, old_end, entity_string):
    """
    Attempts to find the entity in the new text. 
    Returns: (new_span, is_exact_match)
    """
    try:
        # re.escape handles parentheses or special characters in medical names
        matches = list(re.finditer(re.escape(str(entity_string)), new_text))
    except: return None, False
    
    if not matches: return None, False
    
    # 1. Check for Exact Match (Same indices)
    for m in matches:
        if m.start() == old_start and m.end() == old_end:
            return (m.start(), m.end()), True

    # 2. Contextual Realignment
    if len(matches) == 1: return matches[0].span(), False

    context_size = 30
    old_context = old_text[max(0, old_start - context_size) : min(len(old_text), old_end + context_size)]
    best_match, best_score = None, -1
    for m in matches:
        new_start, new_end = m.span()
        new_context = new_text[max(0, new_start - context_size) : min(len(new_text), new_end + context_size)]
        score = SequenceMatcher(None, old_context, new_context).ratio()
        if score > best_score:
            best_score, best_match = score, (new_start, new_end)
            
    return (best_match, False) if best_score > 0.6 else (None, False)

def main():
    print(f"{'='*100}\n🌍 DOCUMENT-LEVEL CONSOLIDATION: OPTIMIZING ANCHOR SELECTION PER ID\n{'='*100}")
    lang_dirs = sorted(glob.glob(os.path.join(RAW_DATA_DIR, "MultiClinNER-*")))
    final_results = []

    for lang_dir in lang_dirs:
        lang = os.path.basename(lang_dir).split('-')[-1]
        print(f"\n▶️ PROCESSING: {lang.upper()}")
        
        # Load all TSVs for the language
        dfs = {}
        original_total = 0
        all_doc_ids = set()
        for ent in ENTITIES:
            path = os.path.join(lang_dir, f"MultiClinNER-{lang}-train", f"MultiClinNER-{lang}-train-{ent}", "tsv", f"MultiClinNER-{lang}-train-{ent}.tsv")
            if os.path.exists(path):
                df = pd.read_csv(path, sep='\t', names=['filename', 'label', 'start', 'end', 'text'], skiprows=1)
                df['doc_id'] = df['filename'].apply(lambda x: str(x).split('-')[-1])
                dfs[ent] = df
                original_total += len(df)
                all_doc_ids.update(df['doc_id'].unique())

        if len(dfs) < 3: continue

        lang_gold_consensus = 0
        lang_rescued = 0
        lang_lost = 0
        
        # Track anchor wins for stats
        anchor_stats = {ent: 0 for ent in ENTITIES}

        for doc_id in sorted(list(all_doc_ids)):
            # Load all 3 text versions
            txts = {}
            for ent in ENTITIES:
                p = os.path.join(lang_dir, f"MultiClinNER-{lang}-train", f"MultiClinNER-{lang}-train-{ent}", "txt", f"MultiClinNER-{lang}-train-{ent}-{doc_id}.txt")
                if os.path.exists(p):
                    with open(p, 'r', encoding='utf-8-sig', newline='') as f:
                        txts[ent] = f.read()
            
            if len(txts) < 3: continue

            # --- DOCUMENT LEVEL EXPERIMENT ---
            # Try each version as the anchor and see which one keeps the most labels
            doc_results = []
            for potential_anchor in ENTITIES:
                anchor_text = txts[potential_anchor]
                gold, rescued, lost = 0, 0, 0
                
                for source_ent in ENTITIES:
                    source_df = dfs[source_ent][dfs[source_ent]['doc_id'] == doc_id]
                    source_text = txts[source_ent]
                    
                    for _, row in source_df.iterrows():
                        new_span, is_exact = realign_span(source_text, anchor_text, int(row['start']), int(row['end']), row['text'])
                        if is_exact: gold += 1
                        elif new_span: rescued += 1
                        else: lost += 1
                
                doc_results.append({
                    "anchor": potential_anchor,
                    "valid": gold + rescued,
                    "gold": gold,
                    "rescued": rescued,
                    "lost": lost
                })
            
            # Pick the best anchor for this specific document
            best_pick = max(doc_results, key=lambda x: x['valid'])
            
            # Update global language stats
            lang_gold_consensus += best_pick['gold']
            lang_rescued += best_pick['rescued']
            lang_lost += best_pick['lost']
            anchor_stats[best_pick['anchor']] += 1

        final_valid = lang_gold_consensus + lang_rescued
        retention = (final_valid / original_total) * 100
        
        print(f"   📊 Results: Retention: {retention:.2f}% ({final_valid}/{original_total})")
        print(f"   ⚓ Anchor Wins: {anchor_stats}")
        print(f"   ✨ Gold Consensus: {lang_gold_consensus} | 🩹 Rescued: {lang_rescued} | 💨 Lost: {lang_lost}")

        final_results.append({
            "Lang": lang.upper(),
            "Expected": original_total,
            "Valid": final_valid,
            "Retention %": round(retention, 2),
            "Best Anchor Split": f"S:{anchor_stats['symptom']} D:{anchor_stats['disease']} P:{anchor_stats['procedure']}"
        })

    print(f"\n{'='*100}\n🏆 FINAL DOCUMENT-LEVEL CONSOLIDATION SUMMARY\n{'='*100}")
    summary_df = pd.DataFrame(final_results)
    print(summary_df.to_string(index=False))

if __name__ == "__main__":
    main()

🌍 DOCUMENT-LEVEL CONSOLIDATION: OPTIMIZING ANCHOR SELECTION PER ID

▶️ PROCESSING: CZ
   📊 Results: Retention: 98.27% (79693/81100)
   ⚓ Anchor Wins: {'symptom': 883, 'disease': 229, 'procedure': 146}
   ✨ Gold Consensus: 51498 | 🩹 Rescued: 28195 | 💨 Lost: 1407

▶️ PROCESSING: EN
   📊 Results: Retention: 99.62% (79014/79316)
   ⚓ Anchor Wins: {'symptom': 1007, 'disease': 113, 'procedure': 138}
   ✨ Gold Consensus: 71240 | 🩹 Rescued: 7774 | 💨 Lost: 302

▶️ PROCESSING: ES
   📊 Results: Retention: 100.00% (83507/83507)
   ⚓ Anchor Wins: {'symptom': 1258, 'disease': 0, 'procedure': 0}
   ✨ Gold Consensus: 83507 | 🩹 Rescued: 0 | 💨 Lost: 0

▶️ PROCESSING: IT
   📊 Results: Retention: 95.07% (77468/81482)
   ⚓ Anchor Wins: {'symptom': 602, 'disease': 213, 'procedure': 443}
   ✨ Gold Consensus: 46419 | 🩹 Rescued: 31049 | 💨 Lost: 4014

▶️ PROCESSING: NL
   📊 Results: Retention: 99.43% (80389/80853)
   ⚓ Anchor Wins: {'symptom': 860, 'disease': 249, 'procedure': 149}
   ✨ Gold Consensus: 63302 | 